# 1. Chiến lược Phân tách Dữ liệu (Data Splitting Strategy)

<div style="font-size: 17px;">

Để chống Rò rỉ dữ liệu (Data Leakage) - nguyên tắc tối thượng của tiền xử lý, nhóm tiến hành chia tập 80% Train - 20% Test với 3 cơ sở kỹ thuật:
  * Gom nhóm động (Dynamic Grouping): Tập dữ liệu tồn tại các danh mục cực hiếm (dưới 5 video/nhóm). Nhóm đã quyết định gộp các danh mục thiểu số này thành một nhóm chung là Others. Thao tác này nhằm thỏa mãn điều kiện biên toán học ($n \ge 2$) của thuật toán phân tầng mà không làm mất mát thông tin.
  * Chia cắt phân tầng (Stratified Sampling): Sau khi gom nhóm, hệ thống ép buộc chia cắt giữ nguyên tỷ lệ phân phối Category (đặc biệt là nhóm Gaming ~63%) ở cả 2 tập.
  * Quy tắc cô lập: Tập Test được niêm phong. Mọi thuật toán ở các bước sau chỉ được học (fit) trên tập Train, rồi mới áp dụng (transform) lên tập Test. (Tham số *random_state = 42* được sử dụng để cố định tính tái lập).

</div>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
file_path = '/kaggle/input/datasets/phn217/ds-raw/ds108_raw.csv' 
df = pd.read_csv(file_path)

print("--- TRAIN-TEST SPLIT ---")

# 1. Phát hiện và xử lý các Category quá hiếm (Dưới 5 video)
category_counts = df['category_name'].value_counts()
rare_categories = category_counts[category_counts < 5].index.tolist()
rare_categories.append('Unknown')

if len(rare_categories) > 0:
    print(f"[!] Phát hiện các danh mục quá hiếm không thể phân tầng: {rare_categories}")
    print("-> Đang gộp các danh mục này thành nhóm 'Others'...\n")
    df['category_name'] = df['category_name'].replace(rare_categories, 'Others')

# 2. Chia cắt Phân tầng
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['category_name'])

print(f"Kích thước tập dữ liệu gốc: {df.shape}")
print(f"-> Kích thước tập Train: {df_train.shape}")
print(f"-> Kích thước tập Test: {df_test.shape}\n")

print("--- KIỂM TRA CHỨNG MINH TỶ LỆ PHÂN TẦNG ---")
print("[Tập TRAIN] Top 5 Category chiếm tỷ trọng:")
print((df_train['category_name'].value_counts(normalize=True) * 100).head(5).round(2).astype(str) + '%')

print("\n[Tập TEST] Top 5 Category chiếm tỷ trọng:")
print((df_test['category_name'].value_counts(normalize=True) * 100).head(5).round(2).astype(str) + '%')

--- TRAIN-TEST SPLIT ---
[!] Phát hiện các danh mục quá hiếm không thể phân tầng: ['Autos & Vehicles', 'Nonprofits & Activism', 'News & Politics', 'Unknown']
-> Đang gộp các danh mục này thành nhóm 'Others'...

Kích thước tập dữ liệu gốc: (6136, 44)
-> Kích thước tập Train: (4908, 44)
-> Kích thước tập Test: (1228, 44)

--- KIỂM TRA CHỨNG MINH TỶ LỆ PHÂN TẦNG ---
[Tập TRAIN] Top 5 Category chiếm tỷ trọng:
category_name
Gaming              63.22%
Entertainment       16.06%
Music               12.84%
People & Blogs       4.01%
Film & Animation     2.28%
Name: proportion, dtype: object

[Tập TEST] Top 5 Category chiếm tỷ trọng:
category_name
Gaming              63.19%
Entertainment       16.04%
Music               12.79%
People & Blogs       3.99%
Film & Animation     2.28%
Name: proportion, dtype: object


# 2. Cứu hộ Dữ liệu nâng cao (Advanced Imputation)

<div style="font-size: 17px;">

Thay vì xóa bỏ hay điền trung bình (làm hỏng phân phối), hệ thống áp dụng quy trình nội suy với 2 luồng xử lý:

* Dữ liệu Định lượng (*view_count*, *like_count*, *comment_count*): Triển khai chiến lược nội suy 2 giai đoạn (Two-Stage Imputation):
  * Giai đoạn 1 (Lỗi API / MNAR): Nhóm video lỗi (subscriber_count = 0) được điền bằng Trung vị theo Danh mục (Category Median) để chặn đứng nguy cơ ngoại suy âm.
  * Giai đoạn 2 (Thiếu tự nhiên / MAR): Nhóm khuyết tự nhiên còn lại được đưa qua thuật toán dự đoán chéo MICE (Iterative Imputer) nhằm bảo toàn cấu trúc dữ liệu gốc.
  * Anti-Data Leakage: Toàn bộ tham số và hệ số hồi quy chỉ được học (fit) trên tập Train, sau đó mới ánh xạ (transform) sang tập Test.

* Dữ liệu Văn bản (*tags*):
  * Triển khai kiến trúc Hybrid LLM-NLP (Llama 3.1 8B Instant qua Groq API) để đọc hiểu ngữ cảnh và tái tạo từ khóa thông minh.
  * Tích hợp Circuit Breaker (Ngắt mạch): Tự động Fallback sang thuật toán NLP cơ bản (Regex/Tokenization) khi API quá tải, đảm bảo lấp đầy 100% dữ liệu mà không làm crash hệ thống.

</div>

In [2]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

cols_to_impute = ['view_count', 'like_count', 'comment_count']

# BƯỚC 0: Convert sang float để tránh lỗi dtype
df_train[cols_to_impute] = df_train[cols_to_impute].astype(float)
df_test[cols_to_impute]  = df_test[cols_to_impute].astype(float)

missing_mask_train = (df_train['view_count'] == 0) | (df_train['like_count'] == 0) | (df_train['comment_count'] == 0)
missing_mask_test  = (df_test['view_count'] == 0) | (df_test['like_count'] == 0) | (df_test['comment_count'] == 0)

print("=== BẮT ĐẦU NỘI SUY DỮ LIỆU TƯƠNG TÁC (TWO-STAGE STRATEGY) ===")

# GIAI ĐOẠN 1: XỬ LÝ NHÓM CRAWL FAILED (sub = 0)
crawl_fail_train = missing_mask_train & (df_train['channel_subscriber_count'] == 0)
crawl_fail_test  = missing_mask_test & (df_test['channel_subscriber_count'] == 0)

# 1. Tính Median theo từng Danh mục (CHỈ TÍNH TRÊN TẬP TRAIN KHÔNG LỖI)
valid_rows_train = df_train[~missing_mask_train]
cat_medians = valid_rows_train.groupby('category_name')[cols_to_impute].median()

# 2. Map (Điền) giá trị vào Train và Test bằng Vector hóa (Nhanh & Mượt hơn vòng lặp for)
for col in cols_to_impute:
    # Áp lên Train
    df_train.loc[crawl_fail_train, col] = df_train.loc[crawl_fail_train, 'category_name'].map(cat_medians[col])
    # Áp lên Test (Chống Data Leakage)
    df_test.loc[crawl_fail_test, col] = df_test.loc[crawl_fail_test, 'category_name'].map(cat_medians[col])

# GIAI ĐOẠN 2: XỬ LÝ NHÓM THIẾU TỰ NHIÊN BẰNG MICE
# Những giá trị 0.0 còn sót lại lúc này chính là nhóm 17% thiếu tự nhiên
df_train[cols_to_impute] = df_train[cols_to_impute].replace(0.0, np.nan)
df_test[cols_to_impute]  = df_test[cols_to_impute].replace(0.0, np.nan)

# Loại biến danh mục dạng số (nếu có)
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != 'category_id']

# Khởi tạo MICE (Thêm min_value=0 để MICE tự động ép đáy, không cần làm thủ công)
mice = IterativeImputer(max_iter=10, random_state=42, min_value=0)

print("Đang huấn luyện MICE trên tập Train...")
df_train[num_cols] = mice.fit_transform(df_train[num_cols])

print("Đang áp dụng mô hình MICE lên tập Test...")
df_test[num_cols]  = mice.transform(df_test[num_cols])

# Hậu xử lý: Làm tròn về số nguyên vì View/Like không thể là số thập phân
df_train[cols_to_impute] = df_train[cols_to_impute].round()
df_test[cols_to_impute]  = df_test[cols_to_impute].round()

print("=> HOÀN TẤT NỘI SUY!")
# Kiểm tra lại thành quả
print("\n[Kiểm tra] Số lượng NaN còn lại trong tập Train:")
print(df_train[cols_to_impute].isnull().sum())
print("\n[Kiểm tra] Số lượng NaN còn lại trong tập Test:")
print(df_test[cols_to_impute].isnull().sum())

=== BẮT ĐẦU NỘI SUY DỮ LIỆU TƯƠNG TÁC (TWO-STAGE STRATEGY) ===
Đang huấn luyện MICE trên tập Train...
Đang áp dụng mô hình MICE lên tập Test...
=> HOÀN TẤT NỘI SUY!

[Kiểm tra] Số lượng NaN còn lại trong tập Train:
view_count       0
like_count       0
comment_count    0
dtype: int64

[Kiểm tra] Số lượng NaN còn lại trong tập Test:
view_count       0
like_count       0
comment_count    0
dtype: int64


In [3]:
print("--- HỢP LƯU DỮ LIỆU NUMERIC (MICE) VÀ TEXT (HYBRID AI) ---")

# 1. Đọc file tags đã chạy xong
file_path = '/kaggle/input/datasets/phn217/ds108-raw-tags-completed/ds108_raw_tags_completed.csv' 
df_tags_completed = pd.read_csv(file_path)

# Bổ sung 'region' và 'crawl_date' vào danh sách cột để làm Siêu Khóa (Composite Key)
cols_to_merge = ['video_id', 'region', 'crawl_date', 'tags', 'tags_source']
df_tags_subset = df_tags_completed[cols_to_merge]

# Xóa các dòng trùng lặp trong tập tag (nếu có) trước khi ghép để tránh Join Explosion
df_tags_subset = df_tags_subset.drop_duplicates(subset=['video_id', 'region', 'crawl_date'])

# 2. Dọn dẹp cột 'tags' rỗng cũ trong tập Train và Test
for df_target in [df_train, df_test]:
    if 'tags' in df_target.columns:
        df_target.drop(columns=['tags'], inplace=True)
    if 'tags_source' in df_target.columns:
        df_target.drop(columns=['tags_source'], inplace=True)

# 3. Tiến hành ghép nối (Left Join) dựa trên SIÊU CHÌA KHÓA
join_keys = ['video_id', 'region', 'crawl_date']
df_train = pd.merge(df_train, df_tags_subset, on=join_keys, how='left')
df_test = pd.merge(df_test, df_tags_subset, on=join_keys, how='left')

print("Đã ghép nối thành công!")
print(f"Kích thước tập Train SAU KHI GHÉP: {df_train.shape}")
print(f"Kích thước tập Test SAU KHI GHÉP : {df_test.shape}")

# 4. Kiểm tra toàn diện
print("\n[Kiểm tra] Số lượng Missing Data trong Train:")
print(df_train[['view_count', 'like_count', 'comment_count', 'tags']].isnull().sum())

print("\n[Kiểm tra] Số lượng Missing Data trong Test:")
print(df_test[['view_count', 'like_count', 'comment_count', 'tags']].isnull().sum())

--- HỢP LƯU DỮ LIỆU NUMERIC (MICE) VÀ TEXT (HYBRID AI) ---
Đã ghép nối thành công!
Kích thước tập Train SAU KHI GHÉP: (4908, 45)
Kích thước tập Test SAU KHI GHÉP : (1228, 45)

[Kiểm tra] Số lượng Missing Data trong Train:
view_count       0
like_count       0
comment_count    0
tags             5
dtype: int64

[Kiểm tra] Số lượng Missing Data trong Test:
view_count       0
like_count       0
comment_count    0
tags             3
dtype: int64


<div style="font-size: 17px;">

Hệ thống còn sót lại 8 bản ghi (5 Train, 3 Test) bị khuyết tags.
* Nguyên nhân: Tiêu đề video chứa ngôn ngữ tượng hình (Hàn, Nhật, Trung...) hoặc Emoji, khiến biểu thức Regex tiếng Anh bỏ qua và tạo ra chuỗi rỗng (NaN).
* Giải pháp: Xử lý nhanh bằng cách lấy trực tiếp tên danh mục (category_name) làm tags dự phòng. Đảm bảo dữ liệu được lấp đầy 100% hoàn hảo.

</div>

In [4]:
print("--- XỬ LÝ NGOẠI LỆ (EDGE CASES) CHO CỘT TAGS ---")

# Dùng chính tên danh mục (Category) để làm tags dự phòng cho 22 dòng ngoại lệ
df_train['tags'] = df_train['tags'].fillna(df_train['category_name'].astype(str))
df_test['tags'] = df_test['tags'].fillna(df_test['category_name'].astype(str))

print("Xử lý thành công!")
print("Số tags rỗng trong Train:", df_train['tags'].isnull().sum())
print("Số tags rỗng trong Test:", df_test['tags'].isnull().sum())

--- XỬ LÝ NGOẠI LỆ (EDGE CASES) CHO CỘT TAGS ---
Xử lý thành công!
Số tags rỗng trong Train: 0
Số tags rỗng trong Test: 0


# 3. Khử trùng lặp Logic Nghiệp vụ (Logical Deduplication)

In [5]:
print("--- KHỬ TRÙNG LẶP LOGIC NGHIỆP VỤ ---")

# Khóa phức hợp định danh một video duy nhất trong 1 ngày tại 1 quốc gia
dup_keys = ['video_id', 'region', 'crawl_date']

# Ghi nhận số dòng ban đầu để báo cáo
len_train_before = len(df_train)
len_test_before = len(df_test)

# 1. Lọc trùng lặp nội bộ trong tập Train
df_train = df_train.drop_duplicates(subset=dup_keys, keep='first')
dropped_train = len_train_before - len(df_train)

# 2. Lọc trùng lặp nội bộ trong tập Test
df_test = df_test.drop_duplicates(subset=dup_keys, keep='first')
dropped_test_internal = len_test_before - len(df_test)

# 3. CHỐT CHẶN DATA LEAKAGE: Khử trùng lặp chéo (Test vs Train)
# Tạo mã định danh duy nhất bằng cách ghép chuỗi: video_id + region + crawl_date
train_keys = df_train['video_id'].astype(str) + "_" + df_train['region'].astype(str) + "_" + df_train['crawl_date'].astype(str)
test_keys = df_test['video_id'].astype(str) + "_" + df_test['region'].astype(str) + "_" + df_test['crawl_date'].astype(str)

# Kiểm tra xem có video nào ở Test bị trùng với Train không
leakage_condition = test_keys.isin(train_keys)
dropped_test_cross = leakage_condition.sum()

if dropped_test_cross > 0:
    print(f"Phát hiện {dropped_test_cross} bản ghi trùng lọt sang tập Test. Đang tiến hành hủy bỏ...")
    # Lọc bỏ các dòng bị rò rỉ (~ là phép phủ định: Giữ lại những dòng KHÔNG nằm trong Train)
    df_test = df_test[~leakage_condition]

# 4. Báo cáo tổng kết
total_dropped = dropped_train + dropped_test_internal + dropped_test_cross

print(f"[Train] Đã dọn dẹp {dropped_train} dòng trùng lặp nội bộ. Số lượng hiện tại: {len(df_train)} dòng.")
print(f"[Test]  Đã dọn dẹp {dropped_test_internal} dòng trùng nội bộ & {dropped_test_cross} dòng rò rỉ. Số lượng hiện tại: {len(df_test)} dòng.")
print(f"Tổng số bản ghi rác đã bị hệ thống triệt tiêu: {total_dropped} dòng.")

--- KHỬ TRÙNG LẶP LOGIC NGHIỆP VỤ ---
Phát hiện 6 bản ghi trùng lọt sang tập Test. Đang tiến hành hủy bỏ...
[Train] Đã dọn dẹp 28 dòng trùng lặp nội bộ. Số lượng hiện tại: 4880 dòng.
[Test]  Đã dọn dẹp 0 dòng trùng nội bộ & 6 dòng rò rỉ. Số lượng hiện tại: 1222 dòng.
Tổng số bản ghi rác đã bị hệ thống triệt tiêu: 34 dòng.
